# 01 · Data pipeline (run once on Colab, GPU runtime)

Downloads every raw input from **public, free sources** and writes the compact
files the modelling stage needs. Runtime ≈ 1.5–2.5 h on a Colab T4 (mostly the
10-K text stream). Re-running is safe: each step caches its outputs.

| Step | Source | Output |
|---|---|---|
| 1 | SEC EDGAR quarterly master index 2009-2023 | `filings.parquet` |
| 2 | derived from step 1 | `events.parquet`, `acquirers.parquet` |
| 3 | SEC `companyfacts.zip` (XBRL) | `fundamentals.parquet` |
| 4 | SEC `submissions.zip` | `companies.parquet` |
| 5 | EDGAR-CORPUS (Loukas et al., 2021), Hugging Face | `text/item1_<year>.parquet` |
| 6 | TF-IDF + multilingual MiniLM embeddings | `tfidf_neighbors.parquet`, `embeddings.npy` |

**Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# 1) Get the code
!git clone https://github.com/<YOUR_GITHUB_USER>/ma-target-radar.git
%cd ma-target-radar
!pip -q install -r requirements.txt

In [ ]:
# 2) Keep outputs on Google Drive so a disconnect does not lose hours of work
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["MTR_ROOT"] = "/content/drive/MyDrive/ma-target-radar"   # data/ lives here
os.makedirs(os.environ["MTR_ROOT"], exist_ok=True)
# SEC requires a real name + e-mail in the User-Agent
os.environ["SEC_USER_AGENT"] = "Joao Miranda your.email@student.ie.edu"

In [ ]:
# 3) Run. If the runtime disconnects, re-run this cell: finished steps are cached.
!MTR_ROOT="$MTR_ROOT" SEC_USER_AGENT="$SEC_USER_AGENT" python run_data.py

When it finishes, `MyDrive/ma-target-radar/data/processed_bundle.zip` contains
everything needed for modelling. Download it (or sync Drive to your laptop) and
continue with `02_models.ipynb` / `python run_models.py`.